In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("Consolidation_Settlement_notebook.ipynb")

# CEE 175: Geotechnical and Geoenvironmental Engineering

> **Alex Frantzis** <br> Cool Student >:3, UC Berkeley

[![License](https://img.shields.io/badge/license-CC%20BY--NC--ND%204.0-blue)](https://creativecommons.org/licenses/by-nc-nd/4.0/)
***

In this assignment, we will model the time-dependent settlement of a saturated clay layer subjected to an applied load. We will implement **Terzaghi's one-dimensional consolidation theory** using the **finite difference method (FDM)** to numerically solve the consolidation equation, track the dissipation of excess pore water pressure over time, and calculate settlement as a function of time.

In [6]:
# Please run this cell, and do not modify the contents

import hashlib
def get_hash(num):
    """Helper function for assessing correctness"""
    return hashlib.md5(str(num).encode()).hexdigest()
    
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

---
## Background: Consolidation Settlement

### What is Consolidation?

When a load is applied to a saturated fine-grained soil (clay or silt), the sudden increase in stress is initially carried entirely by the **pore water**, not the soil skeleton. This is because water is nearly incompressible and cannot instantly escape from the voids. The resulting pressure above the static water pressure is called **excess pore water pressure**, $u_e$.

Over time, as water gradually drains out of the soil voids, the excess pore pressure **dissipates** and the load transfers progressively to the **soil skeleton** as effective stress. This time-dependent process of drainage, effective stress increase, and volume reduction is called **primary consolidation**.

The surface settles as the soil compresses. The rate at which this happens depends on:
- How easily water can flow through the soil (**permeability**, $k$)
- How compressible the soil is (**coefficient of volume compressibility**, $m_v$)
- The drainage path length (**drainage boundary conditions**)

---

### Terzaghi's 1D Consolidation Equation

For a fully saturated, homogeneous clay layer under a uniform load, Terzaghi derived the following **partial differential equation (PDE)** governing the spatial and temporal evolution of excess pore water pressure $u_e(z, t)$:

$$
c_v \frac{\partial^2 u_e}{\partial z^2} = \frac{\partial u_e}{\partial t}
$$

where the **coefficient of consolidation** is:

$$
c_v = \frac{k}{m_v \, \gamma_w}
$$

$$
\begin{aligned}
\text{where:} \\
u_e &= \text{excess pore water pressure (kPa)} \\
c_v &= \text{coefficient of consolidation (m}^2\text{/yr or m}^2\text{/s)} \\
k   &= \text{hydraulic conductivity (m/s)} \\
m_v &= \text{coefficient of volume compressibility (m}^2\text{/kN)} \\
\gamma_w &= \text{unit weight of water (kN/m}^3\text{)}
\end{aligned}
$$

This is the same form as the **heat equation** from physics — a classic diffusion PDE. The pore pressure "diffuses" out of the soil layer over time, just as heat diffuses out of a heated body.

---

### The Finite Difference Method (FDM)

The **finite difference method** is a numerical technique for approximating derivatives. The core idea is simple: instead of the infinitesimally small $dt$ from calculus, use a small but **finite** time step $\Delta t$.

The derivative $\frac{df}{dt}$ is approximated by:

$$
\frac{df}{dt} \approx \frac{f^{n+1} - f^n}{\Delta t}
$$

where $f^n = f(t_n)$ is the known value at the current time step, and $f^{n+1} = f(t_n + \Delta t)$ is the unknown at the next step. This is called the **explicit** (or **forward**) difference because the right-hand side uses only information already known at step $n$.

Rearranging to solve for the unknown at the next step:

$$
f^{n+1} = f^n + \Delta t \cdot \left.\frac{df}{dt}\right|_n
$$

Starting from an initial condition $f^0$, this rule is applied repeatedly — each step uses the previous result to march the solution forward in time.

---

### Warm-up: Approximating $e^t$ with FDM

The exponential function $f(t) = e^{\lambda t}$ satisfies the ODE:

$$
\frac{df}{dt} = \lambda \cdot f, \qquad f(0) = 1
$$

Substituting the forward difference approximation for the derivative:

$$
\frac{f^{n+1} - f^n}{\Delta t} = \lambda \cdot f^n
$$

Solving for $f^{n+1}$:

$$
\boxed{f^{n+1} = f^n + \lambda \cdot \Delta t \cdot f^n}
$$

Starting from $f^0 = 1$, applying this rule repeatedly at each time step approximates $e^{\lambda t}$. The smaller $\Delta t$ is, the more closely the approximation tracks the exact solution — as we will see in the exercise below.

## Question 0: Approximating the Exponential Function

Write a function named `approxExp()` that uses the explicit finite difference update rule derived above to approximate $f(t) = e^{\lambda t}$.

### Input arguments
- `lambda_val` — the growth rate $\lambda$
- `t_max` — the end time of the simulation
- `dt` — the time step size $\Delta t$

### Output
- `t` — NumPy array of time values from $0$ to `t_max` (inclusive) with spacing `dt`
- `f` — NumPy array of the approximated values of $e^{\lambda t}$ at each time step

---
```python
Examples (lambda_val=1):

>>> t, f = approxExp(1, 1.0, 0.5)
>>> f
[1.0, 1.5, 2.25]          # exact e^1.0 ≈ 2.718

>>> t, f = approxExp(1, 2.0, 0.5)
>>> f
[1.0, 1.5, 2.25, 3.375, 5.0625]   # exact e^2.0 ≈ 7.389
```
---
**Hint:** Use `np.arange(0, t_max + dt/2, dt)` to generate the time array. The `dt/2` offset prevents floating-point rounding from accidentally dropping the final point.

In [ ]:
def approxExp(lambda_val, t_max, dt):
    # Create time array from 0 to t_max (inclusive)
    t = np.arange(0, t_max + dt/2, dt)

    # Initialize solution array with zeros
    f = np.zeros(len(t))

    # Set the initial condition: f(0) = e^0 = 1
    f[0] = 1.0  # SOLUTION

    # March forward in time using the explicit FD update rule
    for n in range(len(t) - 1):
        f[n+1] = f[n] + lambda_val * f[n] * dt  # SOLUTION

    return t, f

In [ ]:
# TEST YOUR FUNCTION HERE
lambda_val = 1
t_max = 2.0
dt = 0.5

q0_t, q0_f = approxExp(lambda_val, t_max, dt)
print(f"t values:  {q0_t}")
print(f"f values:  {q0_f}")
print(f"\nFDM final value:   {q0_f[-1]:.5f}")
print(f"Exact e^{t_max}:      {np.exp(lambda_val * t_max):.5f}")

In [ ]:
# Run this cell to visualize your results — try changing dt to see how accuracy changes!
fig, ax = plt.subplots(figsize=(7, 5))

for dt_plot in [0.5, 0.1, 0.01]:
    t_approx, f_approx = approxExp(1, 3, dt_plot)
    ax.plot(t_approx, f_approx, label=f"FDM  ($\\Delta t$ = {dt_plot})")

t_exact = np.linspace(0, 3, 300)
ax.plot(t_exact, np.exp(t_exact), 'k--', linewidth=2, label="Exact:  $e^t$")

ax.set_xlabel("$t$")
ax.set_ylabel("$f(t)$")
ax.set_title("Finite Difference Approximation of $e^t$")
ax.legend()
ax.grid(True, linestyle=":", linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
""" # BEGIN TEST CONFIG
points: 0
failure_message: Make sure to test your function!
""" # END TEST CONFIG

# Check the student ran their function
assert get_hash(type(q0_t)) != '14e736438b115821cbb9b7ac0ba79034'
assert get_hash(type(q0_f)) != '14e736438b115821cbb9b7ac0ba79034'

In [ ]:
""" # BEGIN TEST CONFIG
points: 0
failure_message: Check your initial condition — f[0] should be 1.0.
success_message: Initial condition is correct!
""" # END TEST CONFIG

# Initial condition must be 1.0 regardless of inputs
_t, _f = approxExp(1, 2.0, 0.5)
assert np.isclose(_f[0], 1.0), "f[0] should be 1.0 (since e^0 = 1)"

In [ ]:
""" # BEGIN TEST CONFIG
points: 0
failure_message: Your update rule is incorrect — check the FD formula for f[n+1].
success_message: Update rule is correct!
""" # END TEST CONFIG

# With lambda=1, dt=0.5: each step multiplies by (1 + 0.5) = 1.5
# So f = [1.0, 1.5, 2.25, 3.375, 5.0625] for t_max=2.0
_t, _f = approxExp(1, 2.0, 0.5)
assert np.isclose(_f[1], 1.5),    "After 1 step (lambda=1, dt=0.5): f[1] should be 1.5"
assert np.isclose(_f[2], 2.25),   "After 2 steps (lambda=1, dt=0.5): f[2] should be 2.25"
assert np.isclose(_f[4], 5.0625), "After 4 steps (lambda=1, dt=0.5): f[4] should be 5.0625"

In [ ]:
""" # BEGIN TEST CONFIG
points: 0
failure_message: Check that your function works correctly for different lambda values and step sizes.
success_message: Function handles varying inputs correctly!
""" # END TEST CONFIG

# lambda=2, dt=0.25: each step multiplies by (1 + 2*0.25) = 1.5
# After 4 steps to t=1.0: f[-1] = 1.5^4 = 5.0625
_t, _f = approxExp(2, 1.0, 0.25)
assert np.isclose(_f[-1], 1.5**4), "With lambda=2, dt=0.25, t_max=1.0: f[-1] should be 1.5^4"

# Check time array length is correct: t = [0, dt, 2*dt, ..., t_max]
_t2, _f2 = approxExp(1, 3.0, 0.1)
assert len(_t2) == len(_f2), "t and f arrays must be the same length"
assert np.isclose(_t2[0], 0.0),  "t[0] should be 0.0"
assert np.isclose(_t2[-1], 3.0), "t[-1] should equal t_max"